In [5]:
import os
import json
import numpy as np
from scipy.interpolate import CubicSpline
from sklearn.linear_model import RANSACRegressor, LinearRegression

# ---------- 設定 ----------
json_path = "./testdistance_kernel.json"
output_json_path = "./testdistance_kernel_smoothed.json"

# ---------- ジャンプ補正 ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# ---------- 安定区間抽出 ----------
def find_stable_segments(data, diff_threshold=3.0, min_length=10, min_value=15.0):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    stable_mask = (diffs < diff_threshold) & (data > min_value)

    segments = []
    start = None
    for i, val in enumerate(stable_mask):
        if val:
            if start is None:
                start = i
        else:
            if start is not None and i - start >= min_length:
                segments.append((start, i - 1))
            start = None
    if start is not None and len(data) - start >= min_length:
        segments.append((start, len(data) - 1))
    return segments

# ---------- スプライン補完 ----------
def apply_spline_fit_partial(x_all, data, stable_segments, clip_margin=10.0):
    stable_x = []
    stable_y = []

    median_val = np.median(data)
    min_y = median_val - clip_margin
    max_y = median_val + clip_margin

    for start, end in stable_segments:
        for i in range(start, end + 1):
            if min_y <= data[i] <= max_y:
                stable_x.append(i)
                stable_y.append(data[i])

    if len(stable_x) < 4:
        return data.tolist()

    unique_pairs = list({x: y for x, y in zip(stable_x, stable_y)}.items())
    if len(unique_pairs) < 4:
        return data.tolist()

    unique_pairs.sort()
    sorted_x, sorted_y = zip(*unique_pairs)
    sorted_x = np.array(sorted_x)
    sorted_y = np.clip(np.array(sorted_y), min_y, max_y)

    spline = CubicSpline(sorted_x, sorted_y, bc_type='natural')

    smoothed = data.copy()
    for i in range(len(data)):
        if sorted_x[0] <= i <= sorted_x[-1]:
            smoothed[i] = float(np.clip(spline(i), min_y, max_y))
    return smoothed.tolist()

# ---------- RANSAC補完 ----------
def ransac_fit(data, min_valid=10.0, max_valid=100.0):
    x = np.arange(len(data)).reshape(-1, 1)
    y = np.array(data)

    valid_mask = (y >= min_valid) & (y <= max_valid)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    if len(x_valid) < 2:
        return data.tolist()

    model = RANSACRegressor(estimator=LinearRegression(), min_samples=5, residual_threshold=3.0)
    model.fit(x_valid, y_valid)
    y_pred = model.predict(x)

    return y_pred.tolist()

# ---------- メイン処理 ----------
with open(json_path, encoding="utf-8") as f:
    original_data = json.load(f)

smoothed_data = {}

for scene_id, frame_data in original_data.items():
    frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
    distances = np.array([frame_data[k] for k in frame_keys], dtype=float)
    x_all = np.arange(len(distances))

    if scene_id == "211":
        smoothed_values = ransac_fit(distances)
    else:
        jump_corrected = suppress_jumps(distances)
        segments = find_stable_segments(jump_corrected)
        smoothed_values = apply_spline_fit_partial(x_all, jump_corrected, segments)

    # フレームごとの結果を保存
    smoothed_data[scene_id] = {
        key: round(val, 4) for key, val in zip(frame_keys, smoothed_values)
    }

# ---------- 書き出し ----------
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(smoothed_data, f, ensure_ascii=False, indent=2)

print(f"✅ 全シーンの補完結果を保存しました: {output_json_path}")


✅ 全シーンの補完結果を保存しました: ./testdistance_kernel_smoothed.json


In [6]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# ---------- 設定 ----------
INPUT_JSON = "./testdistance_kernel_smoothed.json"
OUTPUT_JSON = "./test.json"
GRAPH_DIR = "./smoothed_graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)

# ---------- 入力読み込み ----------
with open(INPUT_JSON, encoding="utf-8") as f:
    all_data = json.load(f)

# ---------- 出力用データ ----------
smoothed_data = {}

# ---------- 各シーン処理 ----------
for scene_id, frame_dict in all_data.items():
    frame_keys = sorted(frame_dict.keys())
    x = np.array([int(fid.replace("frame_", "")) for fid in frame_keys])
    y = np.array([frame_dict[fid] for fid in frame_keys])

    # Savitzky-Golay フィルター適用（ウィンドウサイズは調整）
    if len(y) >= 5:
        window_length = min(len(y) // 2 * 2 + 1, 25)  # 奇数に制限
        polyorder = 3 if window_length >= 5 else 2
        y_smooth = savgol_filter(y, window_length, polyorder)
    else:
        y_smooth = y  # データ点が少なすぎる場合はそのまま

    # 辞書に保存（元のkeyに対応）
    smoothed_data[scene_id] = {
        frame_keys[i]: float(round(y_smooth[i], 4)) for i in range(len(frame_keys))
    }

    # グラフ描画・保存
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'o', color='gray', label='Original')
    plt.plot(x, y_smooth, '-', color='green', label='Smoothed')
    plt.title(f"Scene {scene_id} - Smoothed Distance")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(GRAPH_DIR, f"scene{scene_id}.png"))
    plt.close()

# ---------- 新しい JSON 保存 ----------
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(smoothed_data, f, ensure_ascii=False, indent=2)

print("✅ 平滑化完了：JSON保存 ＆ グラフ出力完了")


✅ 平滑化完了：JSON保存 ＆ グラフ出力完了


In [7]:
import os
import json
import numpy as np
from scipy.interpolate import make_interp_spline

# ---------- 設定 ----------
INPUT_JSON = "./test.json"
OUTPUT_JSON = "./test_spline_smoothed.json"

# ---------- データ読み込み ----------
with open(INPUT_JSON, encoding="utf-8") as f:
    all_data = json.load(f)

smoothed_data = {}

# ---------- 各シーンを処理 ----------
for scene_id, frame_dict in all_data.items():
    frame_keys = sorted(frame_dict.keys())
    x = np.array([int(k.replace("frame_", "")) for k in frame_keys])
    y = np.array([frame_dict[k] for k in frame_keys])

    # スプライン補間 → 元のフレーム数に再サンプリング
    if len(x) >= 4:
        x_dense = np.linspace(x.min(), x.max(), len(x) * 5)
        spline = make_interp_spline(x, y, k=3)
        y_dense = spline(x_dense)
        y_resampled = np.interp(x, x_dense, y_dense)
    else:
        y_resampled = y  # 点が少なすぎる場合はそのまま

    # 保存形式に変換（丸めなし）
    smoothed_data[scene_id] = {
        frame_keys[i]: float(y_resampled[i]) for i in range(len(frame_keys))
    }

# ---------- 新しいJSONに保存 ----------
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(smoothed_data, f, ensure_ascii=False, indent=2)

print("✅ 丸めなしで test_spline_smoothed.json に保存しました。")


✅ 丸めなしで test_spline_smoothed.json に保存しました。


In [9]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# ---------- 設定 ----------
ORIGINAL_JSON = "./testdistance_kernel.json"
SMOOTHED_JSON = "./test_spline_smoothed.json"
GRAPH_DIR = "./comparison_graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)

# ---------- JSON読み込み ----------
with open(ORIGINAL_JSON, encoding="utf-8") as f:
    original_data = json.load(f)

with open(SMOOTHED_JSON, encoding="utf-8") as f:
    smoothed_data = json.load(f)

# ---------- 比較グラフ描画 ----------
for scene_id in original_data:
    if scene_id not in smoothed_data:
        continue

    # 共通のフレームを取得（順序保証）
    orig_frames = sorted(original_data[scene_id].keys())
    smooth_frames = sorted(smoothed_data[scene_id].keys())
    common_frames = sorted(set(orig_frames) & set(smooth_frames))
    if len(common_frames) < 2:
        continue

    x = np.array([int(f.replace("frame_", "")) for f in common_frames])
    y_orig = np.array([original_data[scene_id][f] for f in common_frames])
    y_smooth = np.array([smoothed_data[scene_id][f] for f in common_frames])

    # 描画
    plt.figure(figsize=(10, 6))
    plt.plot(x, y_orig, 'o-', color='gray', label='Original')
    plt.plot(x, y_smooth, 'o-', color='green', label='Spline Smoothed')
    plt.title(f"Scene {scene_id} - Distance Comparison")
    plt.xlabel("Frame Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(GRAPH_DIR, f"scene{scene_id}.png"))
    plt.close()

print("✅ 比較グラフを ./comparison_graphs に保存しました。")


✅ 比較グラフを ./comparison_graphs に保存しました。
